In [ ]:
# Cell 1 — Parameters
TENOR = '10Y'
COUNTRIES = {
    'Peru': df_perugb_cmt,
    'Mexico': df_mbono_cmt,
    'Colombia': df_coltes_cmt,
    'Chile': df_btpcl_cmt,
}
FOCUS_COUNTRY = 'Peru'
LOOKBACK_START = '2022-01-01'  # None = full history
N_PCS = 2  # None = auto via 90% cumvar
SD_WINDOW = 252
SD_BANDS = [1.25, 1.65]
ROLLING_WINDOWS = [5, 10, 20]
ROLL_DISPLAY = [5, 20]
TRAIL_WINDOW = 60

In [ ]:
# Cell 2 — Build Spread Panel
import pandas as pd
import numpy as np

def build_spread_panel(countries, tenor, ust_df):
    ust = ust_df.set_index('Fecha')[tenor].rename('UST')
    frames = {}
    for name, df in countries.items():
        s = df.set_index('Fecha')[tenor]
        aligned = pd.concat([s, ust], axis=1).dropna()
        frames[name] = (aligned[tenor] - aligned['UST']) * 100
    df_out = pd.DataFrame(frames)
    df_out.index.name = 'Fecha'
    return df_out

df_spreads = build_spread_panel(COUNTRIES, TENOR, df_ust_cmt)
print(f'Shape: {df_spreads.shape}')
print(f'Date range: {df_spreads.index.min().date()} — {df_spreads.index.max().date()}')

In [ ]:
# Cell 3 — PCA Engine + Decompose

def run_pca(df_spreads, lookback_start, n_pcs):
    df = df_spreads.copy()
    if lookback_start is not None:
        df = df[df.index >= lookback_start]
    df = df.dropna()
    means = df.mean()
    demeaned = df - means
    cov = demeaned.cov().values
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    # sort descending
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]
    var_explained = eigenvalues / eigenvalues.sum()
    cum_var = np.cumsum(var_explained)
    if n_pcs is None:
        n_pcs = int(np.searchsorted(cum_var, 0.90)) + 1
    loadings = pd.DataFrame(
        eigenvectors[:, :n_pcs],
        index=df.columns,
        columns=[f'PC{i+1}' for i in range(n_pcs)]
    )
    scores = pd.DataFrame(
        demeaned.values @ eigenvectors[:, :n_pcs],
        index=df.index,
        columns=[f'PC{i+1}' for i in range(n_pcs)]
    )
    return {
        'means': means,
        'loadings': loadings,
        'scores': scores,
        'eigenvalues': eigenvalues[:n_pcs],
        'var_explained': var_explained[:n_pcs],
        'cum_var_explained': cum_var[:n_pcs],
        'n_pcs': n_pcs,
    }

def decompose(pca, df_spreads, lookback_start=None):
    df = df_spreads.copy()
    if lookback_start is not None:
        df = df[df.index >= lookback_start]
    df = df.dropna()
    demeaned = df - pca['means']
    demeaned = demeaned.dropna()
    loadings = pca['loadings']
    scores = pd.DataFrame(
        demeaned.values @ loadings.values,
        index=demeaned.index,
        columns=loadings.columns
    )
    fitted = pd.DataFrame(
        scores.values @ loadings.values.T + pca['means'].values,
        index=demeaned.index,
        columns=df.columns
    )
    residuals = df.loc[demeaned.index] - fitted
    contributions = {}
    for country in df.columns:
        contrib = pd.DataFrame(index=demeaned.index)
        for pc in loadings.columns:
            contrib[pc] = scores[pc] * loadings.loc[country, pc]
        contributions[country] = contrib
    return {
        'fitted': fitted,
        'residuals': residuals,
        'contributions': contributions,
        'scores_full': scores,
    }

pca = run_pca(df_spreads, LOOKBACK_START, N_PCS)
decomp = decompose(pca, df_spreads, LOOKBACK_START)

print(f'PCs retained: {pca["n_pcs"]}')
print(f'Variance explained: {[f"{v:.1%}" for v in pca["var_explained"]]}')
print(f'Fitted date range: {decomp["fitted"].index.min().date()} — {decomp["fitted"].index.max().date()}')
print(f'Residuals shape: {decomp["residuals"].shape}')

In [ ]:
# Cell 4 — Country Selection Diagnostic
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

countries_all = {
    'Peru': df_perugb_cmt,
    'Mexico': df_mbono_cmt,
    'Colombia': df_coltes_cmt,
    'Chile': df_btpcl_cmt,
}
countries_ex_mexico = {
    'Peru': df_perugb_cmt,
    'Colombia': df_coltes_cmt,
    'Chile': df_btpcl_cmt,
}

sp_all = build_spread_panel(countries_all, TENOR, df_ust_cmt)
sp_exmx = build_spread_panel(countries_ex_mexico, TENOR, df_ust_cmt)

pca_all = run_pca(sp_all, LOOKBACK_START, N_PCS)
pca_exmx = run_pca(sp_exmx, LOOKBACK_START, N_PCS)

# side-by-side variance and loadings
print('=== All Countries ===')
for i, (v, cv) in enumerate(zip(pca_all['var_explained'], pca_all['cum_var_explained'])):
    print(f'  PC{i+1}: {v:.1%}  cumulative: {cv:.1%}')
print(pca_all['loadings'].to_string(float_format=lambda x: f'{x:.4f}'))

print('\n=== Ex-Mexico ===')
for i, (v, cv) in enumerate(zip(pca_exmx['var_explained'], pca_exmx['cum_var_explained'])):
    print(f'  PC{i+1}: {v:.1%}  cumulative: {cv:.1%}')
print(pca_exmx['loadings'].to_string(float_format=lambda x: f'{x:.4f}'))

# Peru residual comparison
decomp_all = decompose(pca_all, sp_all, LOOKBACK_START)
decomp_exmx = decompose(pca_exmx, sp_exmx, LOOKBACK_START)

for label, d in [('All Countries', decomp_all), ('Ex-Mexico', decomp_exmx)]:
    r = d['residuals']['Peru']
    ac1 = r.autocorr(1)
    ac5 = r.autocorr(5)
    print(f'\nPeru residuals [{label}]: Std={r.std():.2f}  AC(1)={ac1:.3f}  AC(5)={ac5:.3f}  Min={r.min():.1f}  Max={r.max():.1f}')

# grouped bar charts
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, label, pca_r in [(axes[0], 'All Countries', pca_all), (axes[1], 'Ex-Mexico', pca_exmx)]:
    ld = pca_r['loadings']
    n_countries = len(ld)
    x = np.arange(n_countries)
    n_pcs_r = pca_r['n_pcs']
    width = 0.8 / n_pcs_r
    colors = plt.cm.tab10.colors
    for i, pc in enumerate(ld.columns):
        vals = ld[pc].values
        bars = ax.bar(x + i * width - (n_pcs_r - 1) * width / 2, vals, width, label=pc, color=colors[i])
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005 * np.sign(val + 1e-9),
                    f'{val:.2f}', ha='center', va='bottom' if val >= 0 else 'top', fontsize=8)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(ld.index, rotation=15)
    ax.set_title(f'PCA Loadings — {label}')
    ax.legend()

plt.suptitle(f'Country Selection Diagnostic ({TENOR})', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 5 — Factor Returns & Rolling Sums
factor_returns = decomp['scores_full'].diff().dropna()
roll_factor = {}
for w in ROLLING_WINDOWS:
    roll_factor[w] = factor_returns.rolling(w).sum()

print(f'Factor returns shape: {factor_returns.shape}')
print(f'Rolling windows computed: {ROLLING_WINDOWS}')

In [ ]:
# Cell 6 — Variance Explained Table + Bar Chart
import matplotlib.pyplot as plt

# table
print(f'{"PC":<6}{"Var Explained":>15}{"Cum Var Explained":>20}')
print('-' * 42)
for i, (v, cv) in enumerate(zip(pca['var_explained'], pca['cum_var_explained'])):
    print(f'PC{i+1:<4}{v*100:>14.2f}%{cv*100:>19.2f}%')

# bar chart
fig, ax = plt.subplots(figsize=(7, 4))
pcs = [f'PC{i+1}' for i in range(pca['n_pcs'])]
vals = pca['var_explained'] * 100
bars = ax.bar(pcs, vals, color='steelblue')
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=10)
ax.set_ylabel('Variance Explained (%)')
ax.set_title(f'PCA Variance Explained ({TENOR})')
plt.tight_layout()
plt.show()

# loadings matrix
print('\nLoadings:')
print(pca['loadings'].to_string(float_format=lambda x: f'{x:.4f}'))

In [ ]:
# Cell 7 — Loadings Bar Chart
fig, ax = plt.subplots(figsize=(9, 5))
ld = pca['loadings']
n_countries = len(ld)
x = np.arange(n_countries)
n_pcs_plot = pca['n_pcs']
width = 0.8 / n_pcs_plot
colors = plt.cm.tab10.colors

for i, pc in enumerate(ld.columns):
    vals = ld[pc].values
    bars = ax.bar(x + i * width - (n_pcs_plot - 1) * width / 2, vals, width,
                  label=pc, color=colors[i])
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.005 * (1 if val >= 0 else -1),
                f'{val:.2f}', ha='center',
                va='bottom' if val >= 0 else 'top', fontsize=9)

ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(ld.index, rotation=15)
ax.set_ylabel('Loading')
ax.set_title(f'PCA Loadings by Country ({TENOR})')
ax.legend(title='PC')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 8 — Loading Magnitude vs Volatility
df_lb = df_spreads.copy()
if LOOKBACK_START is not None:
    df_lb = df_lb[df_lb.index >= LOOKBACK_START]
df_lb = df_lb.dropna()

daily_chg_sd = df_lb.diff().std()
pc1_abs = pca['loadings']['PC1'].abs()

# normalize to largest country
max_sd = daily_chg_sd.max()
max_loading = pc1_abs.max()
vol_ratio = daily_chg_sd / max_sd
loading_ratio = pc1_abs / max_loading
ratio_match = vol_ratio / loading_ratio

print(f'{"Country":<12}{"Daily Chg SD":>14}{"|PC1 Loading|":>15}{"Vol Ratio":>12}{"Loading Ratio":>15}{"Ratio Match":>13}')
print('-' * 82)
for country in df_lb.columns:
    print(f'{country:<12}{daily_chg_sd[country]:>13.3f} '
          f'{pc1_abs[country]:>14.4f} '
          f'{vol_ratio[country]:>11.4f} '
          f'{loading_ratio[country]:>14.4f} '
          f'{ratio_match[country]:>12.4f}')

print('\nRatio Match near 1.0 confirms loadings are volatility-driven.')

In [ ]:
# Cell 9 — Correlation-Based PCA Diagnostic

def run_pca_corr(df_spreads, lookback_start, n_pcs):
    df = df_spreads.copy()
    if lookback_start is not None:
        df = df[df.index >= lookback_start]
    df = df.dropna()
    standardized = (df - df.mean()) / df.std()
    cov = standardized.cov().values
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]
    var_explained = eigenvalues / eigenvalues.sum()
    cum_var = np.cumsum(var_explained)
    if n_pcs is None:
        n_pcs = int(np.searchsorted(cum_var, 0.90)) + 1
    loadings = pd.DataFrame(
        eigenvectors[:, :n_pcs],
        index=df.columns,
        columns=[f'PC{i+1}' for i in range(n_pcs)]
    )
    return {
        'loadings': loadings,
        'var_explained': var_explained[:n_pcs],
        'n_pcs': n_pcs,
    }

pca_corr = run_pca_corr(df_spreads, LOOKBACK_START, N_PCS)

# side-by-side variance explained
print(f'{"PC":<6}{"Cov Var%":>10}{"Corr Var%":>12}')
print('-' * 30)
for i in range(max(pca['n_pcs'], pca_corr['n_pcs'])):
    cv = pca['var_explained'][i] * 100 if i < pca['n_pcs'] else float('nan')
    cc = pca_corr['var_explained'][i] * 100 if i < pca_corr['n_pcs'] else float('nan')
    print(f'PC{i+1:<4}{cv:>9.2f}%{cc:>11.2f}%')

# loadings side by side
print('\nCovariance Loadings:')
print(pca['loadings'].to_string(float_format=lambda x: f'{x:.4f}'))
print('\nCorrelation Loadings:')
print(pca_corr['loadings'].to_string(float_format=lambda x: f'{x:.4f}'))

# two grouped bar charts
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, label, pca_r in [(axes[0], 'Covariance', pca), (axes[1], 'Correlation', pca_corr)]:
    ld = pca_r['loadings']
    n_c = len(ld)
    x = np.arange(n_c)
    n_pcs_r = pca_r['n_pcs']
    width = 0.8 / n_pcs_r
    colors = plt.cm.tab10.colors
    for i, pc in enumerate(ld.columns):
        vals = ld[pc].values
        bars = ax.bar(x + i * width - (n_pcs_r - 1) * width / 2, vals, width,
                      label=pc, color=colors[i])
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.005 * (1 if val >= 0 else -1),
                    f'{val:.2f}', ha='center',
                    va='bottom' if val >= 0 else 'top', fontsize=8)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(ld.index, rotation=15)
    ax.set_title(f'{label}-Based Loadings')
    ax.legend()

plt.suptitle(f'PCA Loadings — Covariance vs Correlation ({TENOR})', fontsize=13)
plt.tight_layout()
plt.show()
print('Note: Correlation loadings are more balanced (volatility removed). Covariance PCA remains the working model.')

In [ ]:
# Cell 10 — Main Diagnostic Chart (FOCUS_COUNTRY)
import matplotlib.patches as mpatches

def shade_bands(ax, dates, series, roll_sd, bands):
    # shade between inner and outer and beyond outer
    inner, outer = bands[0], bands[1]
    for i in range(len(dates) - 1):
        v = series.iloc[i]
        sd = roll_sd.iloc[i]
        if pd.isna(v) or pd.isna(sd) or sd == 0:
            continue
        z = v / sd
        x0, x1 = dates[i], dates[i + 1]
        if z > outer:
            ax.axvspan(x0, x1, color='red', alpha=0.3, linewidth=0)
        elif z > inner:
            ax.axvspan(x0, x1, color='orange', alpha=0.2, linewidth=0)
        elif z < -outer:
            ax.axvspan(x0, x1, color='darkblue', alpha=0.3, linewidth=0)
        elif z < -inner:
            ax.axvspan(x0, x1, color='lightblue', alpha=0.4, linewidth=0)

country = FOCUS_COUNTRY
actual = df_spreads[country].dropna()
fitted = decomp['fitted'][country]
residual = decomp['residuals'][country]
contrib = decomp['contributions'][country]
mean_val = pca['means'][country]
roll_sd = residual.rolling(SD_WINDOW).std()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

# top panel: stacked area of contributions + mean as base
pcs = contrib.columns.tolist()
colors_area = plt.cm.tab10.colors
base = pd.Series(mean_val, index=contrib.index)
bottom_pos = base.copy()
bottom_neg = base.copy()
ax1.fill_between(contrib.index, 0, base, alpha=0.15, color='gray', label=f'Mean ({mean_val:.1f} bps)')
for i, pc in enumerate(pcs):
    vals = contrib[pc]
    upper_pos = bottom_pos + vals.clip(lower=0)
    upper_neg = bottom_neg + vals.clip(upper=0)
    ax1.fill_between(contrib.index, bottom_pos, upper_pos, alpha=0.5,
                     color=colors_area[i], label=pc)
    ax1.fill_between(contrib.index, bottom_neg, upper_neg, alpha=0.5,
                     color=colors_area[i])
    bottom_pos = upper_pos
    bottom_neg = upper_neg
ax1.plot(actual.index, actual, color='black', linewidth=1.5, label='Actual')
ax1.set_ylabel('Spread (bps)')
ax1.legend(loc='upper left', fontsize=8)
ax1.set_title(
    f'{FOCUS_COUNTRY} | {TENOR} Spread vs UST | '
    f'PCA from {LOOKBACK_START if LOOKBACK_START else "full history"}'
)

# bottom panel: residual + SD bands + shading
dates = residual.index.tolist()
shade_bands(ax2, dates, residual, roll_sd, SD_BANDS)
ax2.plot(residual.index, residual, color='black', linewidth=1.2, label='Residual')
ax2.axhline(0, color='black', linewidth=0.6)
band_colors = ['darkorange', 'red']
for thresh, col in zip(SD_BANDS, band_colors):
    ub = roll_sd * thresh
    lb = -roll_sd * thresh
    ax2.plot(roll_sd.index, ub, linestyle='--', color=col, linewidth=0.9,
             label=f'+{thresh}σ')
    ax2.plot(roll_sd.index, lb, linestyle='--', color=col, linewidth=0.9,
             label=f'-{thresh}σ')
ax2.set_ylabel('Residual (bps)')
ax2.legend(loc='upper left', fontsize=8, ncol=3)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 11 — All Countries Residual Dashboard
countries_list = list(decomp['residuals'].columns)
n_c = len(countries_list)
fig, axes = plt.subplots(n_c, 1, figsize=(14, 4 * n_c), sharex=True)
if n_c == 1:
    axes = [axes]

band_colors = ['darkorange', 'red']

for ax, cname in zip(axes, countries_list):
    resid = decomp['residuals'][cname]
    roll_sd = resid.rolling(SD_WINDOW).std()
    dates = resid.index.tolist()
    shade_bands(ax, dates, resid, roll_sd, SD_BANDS)
    ax.plot(resid.index, resid, color='black', linewidth=1.1)
    ax.axhline(0, color='black', linewidth=0.6)
    for thresh, col in zip(SD_BANDS, band_colors):
        ax.plot(roll_sd.index, roll_sd * thresh, linestyle='--', color=col, linewidth=0.8)
        ax.plot(roll_sd.index, -roll_sd * thresh, linestyle='--', color=col, linewidth=0.8)
    ax.set_ylabel('Residual (bps)')
    ax.set_title(cname)

plt.suptitle(f'All Countries — {TENOR} Spread Residuals vs UST', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 12 — PC Scores with Regime Shading
scores = decomp['scores_full']
n_pcs_plot = pca['n_pcs']
fig, axes = plt.subplots(n_pcs_plot, 1, figsize=(14, 4 * n_pcs_plot), sharex=True)
if n_pcs_plot == 1:
    axes = [axes]

for ax, pc in zip(axes, scores.columns):
    s = scores[pc]
    trail = s.rolling(TRAIL_WINDOW).mean()
    # regime shading
    for i in range(len(s) - 1):
        if pd.isna(trail.iloc[i]):
            continue
        x0, x1 = s.index[i], s.index[i + 1]
        if s.iloc[i] > trail.iloc[i]:
            ax.axvspan(x0, x1, color='green', alpha=0.15, linewidth=0)
        else:
            ax.axvspan(x0, x1, color='red', alpha=0.15, linewidth=0)
    ax.plot(s.index, s, color='steelblue', linewidth=1.2, label=pc)
    ax.plot(trail.index, trail, color='navy', linestyle='--', linewidth=1.0,
            label=f'{TRAIL_WINDOW}d trailing mean')
    ax.axhline(0, color='black', linewidth=0.6)
    ax.set_ylabel('Score')
    ax.set_title(pc)
    ax.legend(fontsize=8)

plt.suptitle(f'PC Scores — Levels and Regime ({TENOR})', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 13 — Factor Return Momentum
n_pcs_plot = pca['n_pcs']
fig, axes = plt.subplots(n_pcs_plot, 1, figsize=(14, 4 * n_pcs_plot), sharex=True)
if n_pcs_plot == 1:
    axes = [axes]

roll_colors = plt.cm.tab10.colors

for ax, pc in zip(axes, factor_returns.columns):
    for j, w in enumerate(ROLL_DISPLAY):
        ax.plot(roll_factor[w].index, roll_factor[w][pc],
                color=roll_colors[j], linewidth=1.2, label=f'{w}d')
    ax.axhline(0, color='black', linewidth=0.6)
    ax.set_ylabel('Rolling Sum')
    ax.set_title(pc)
    ax.legend(fontsize=8)

plt.suptitle(f'Factor Return Momentum — Rolling Sums ({TENOR})', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 14 — Current Snapshot Table
last_date = decomp['residuals'].index[-1]
resid_last = decomp['residuals'].loc[last_date]
actual_last = df_spreads.loc[last_date] if last_date in df_spreads.index else decomp['fitted'].loc[last_date] + resid_last
fitted_last = decomp['fitted'].loc[last_date]

# rolling SD for z-score
roll_sd_all = {c: decomp['residuals'][c].rolling(SD_WINDOW).std() for c in decomp['residuals'].columns}

pcs = pca['loadings'].columns.tolist()

print(f'=== Snapshot as of {last_date.date()} | Tenor: {TENOR} ===\n')

# header
pc_headers = '  '.join([f'{pc:>10}' for pc in pcs])
print(f'{"Country":<12}{"Actual":>10}{"Fitted":>10}{"Residual":>11}{"Z-Score":>10}  {pc_headers}')
print('-' * (12 + 10 + 10 + 11 + 10 + 2 + 12 * len(pcs)))

for country in decomp['residuals'].columns:
    act = actual_last[country] if country in actual_last.index else float('nan')
    fit = fitted_last[country]
    res = resid_last[country]
    sd_val = roll_sd_all[country].iloc[-1]
    z = res / sd_val if not pd.isna(sd_val) and sd_val != 0 else float('nan')
    contrib_last = decomp['contributions'][country].loc[last_date]
    pc_vals = '  '.join([f'{contrib_last[pc]:>10.1f}' for pc in pcs])
    print(f'{country:<12}{act:>10.1f}{fit:>10.1f}{res:>11.1f}{z:>10.2f}  {pc_vals}')

In [ ]:
# Part 2 — Additional Parameters
ATTRIB_WINDOWS = [5, 10, 20]
FWD_WINDOWS = [5, 10, 20, 40]
ZSCORE_BINS = [-999, -1.65, -1.25, 0, 1.25, 1.65, 999]
ZSCORE_LABELS = ['<-1.65', '-1.65:-1.25', '-1.25:0', '0:1.25', '1.25:1.65', '>1.65']

In [ ]:
# Cell 15 — Period Attribution Table
import pandas as pd
import numpy as np

def period_attribution(decomp, df_spreads, country, windows, lookback_start=None):
    actual = df_spreads[country].copy()
    resid = decomp['residuals'][country].copy()
    contrib = decomp['contributions'][country].copy()
    if lookback_start is not None:
        actual = actual[actual.index >= lookback_start]
        resid = resid[resid.index >= lookback_start]
        contrib = contrib[contrib.index >= lookback_start]
    # align on common index
    idx = resid.index
    actual = actual.reindex(idx).dropna()
    resid = resid.reindex(actual.index)
    contrib = contrib.reindex(actual.index)
    rows = []
    for w in windows:
        if len(actual) < w + 1:
            continue
        act_chg = actual.iloc[-1] - actual.iloc[-(w + 1)]
        resid_chg = resid.iloc[-1] - resid.iloc[-(w + 1)]
        row = {'Window': f'{w}d', 'Actual Chg': act_chg}
        pc_sum = 0.0
        for pc in contrib.columns:
            c_chg = contrib[pc].iloc[-1] - contrib[pc].iloc[-(w + 1)]
            row[f'{pc} Contrib'] = c_chg
            pc_sum += c_chg
        row['Residual Chg'] = resid_chg
        row['Residual Chg %'] = (resid_chg / abs(act_chg) * 100) if act_chg != 0 else float('nan')
        rows.append(row)
    return pd.DataFrame(rows).set_index('Window')

last_date = decomp['residuals'].index[-1]
print(f'=== Period Attribution: {FOCUS_COUNTRY} | {TENOR} | as of {last_date.date()} ===\n')
attr = period_attribution(decomp, df_spreads, FOCUS_COUNTRY, ATTRIB_WINDOWS, LOOKBACK_START)

# format output
pc_cols = [c for c in attr.columns if 'Contrib' in c and c != 'Residual Chg %']
fmt_cols = ['Actual Chg'] + pc_cols + ['Residual Chg']
print(attr[fmt_cols + ['Residual Chg %']].to_string(
    float_format=lambda x: f'{x:.1f}',
    formatters={'Residual Chg %': lambda x: f'{x:.1f}%' if not pd.isna(x) else 'n/a'}
))

In [ ]:
# Cell 16 — Residual Persistence Diagnostic
import matplotlib.pyplot as plt
from scipy import stats

resid_fc = decomp['residuals'][FOCUS_COUNTRY].dropna()

# --- half-life via OLS regression ---
y = resid_fc.values[1:]
x = resid_fc.values[:-1]
slope, intercept, r, p, se = stats.linregress(x, y)
half_life = -np.log(2) / np.log(abs(slope))
print(f'OLS persistence coefficient: {slope:.4f}')
print(f'Estimated half-life: {half_life:.1f} business days')

# --- rolling 252d half-life ---
roll_hl = pd.Series(index=resid_fc.index, dtype=float)
for i in range(252, len(resid_fc)):
    window_vals = resid_fc.iloc[i - 252:i].values
    y_w = window_vals[1:]
    x_w = window_vals[:-1]
    sl, *_ = stats.linregress(x_w, y_w)
    if sl > 0 and sl < 1:
        roll_hl.iloc[i] = -np.log(2) / np.log(sl)
    else:
        roll_hl.iloc[i] = float('nan')

fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=False)

# top: residual + 60d rolling mean
roll_mean_60 = resid_fc.rolling(60).mean()
axes[0].plot(resid_fc.index, resid_fc, color='black', linewidth=0.9, label='Residual')
axes[0].plot(roll_mean_60.index, roll_mean_60, color='steelblue', linewidth=1.5,
             linestyle='--', label='60d Rolling Mean')
axes[0].axhline(0, color='gray', linewidth=0.6)
axes[0].set_ylabel('Residual (bps)')
axes[0].set_title(f'Residual Persistence — {FOCUS_COUNTRY} ({TENOR})')
axes[0].legend(fontsize=9)

# middle: OLS scatter with fit line
axes[1].scatter(x, y, alpha=0.2, s=8, color='steelblue')
x_line = np.linspace(x.min(), x.max(), 200)
axes[1].plot(x_line, slope * x_line + intercept, color='red', linewidth=1.5,
             label=f'slope={slope:.4f}  HL={half_life:.1f}d')
axes[1].set_xlabel('Residual(t)')
axes[1].set_ylabel('Residual(t+1)')
axes[1].set_title('AR(1) Regression — Mean Reversion')
axes[1].legend(fontsize=9)

# bottom: rolling 252d half-life
axes[2].plot(roll_hl.index, roll_hl, color='darkorange', linewidth=1.2)
axes[2].axhline(half_life, color='black', linestyle='--', linewidth=0.8,
                label=f'Full-sample HL={half_life:.1f}d')
axes[2].set_ylabel('Half-life (days)')
axes[2].set_title('Rolling 252d Half-Life')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 17 — Forward-Return Signal Analysis
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

def forward_return_analysis(decomp, df_spreads, country, fwd_windows, zscore_bins,
                            zscore_labels, sd_window, lookback_start=None):
    resid = decomp['residuals'][country].copy()
    actual = df_spreads[country].copy()
    if lookback_start is not None:
        resid = resid[resid.index >= lookback_start]
        actual = actual[actual.index >= lookback_start]
    actual = actual.reindex(resid.index)
    roll_sd = resid.rolling(sd_window).std()
    zscore = resid / roll_sd
    bucket = pd.cut(zscore, bins=zscore_bins, labels=zscore_labels)
    # middle buckets where no directional expectation
    neutral_buckets = {'-1.25:0', '0:1.25'}
    # positive-z buckets: mean reversion = negative fwd change
    pos_buckets = {'>1.65', '1.25:1.65'}
    results = {}
    for w in fwd_windows:
        fwd_resid = resid.shift(-w) - resid
        fwd_spread = actual.shift(-w) - actual
        rows = []
        for lbl in zscore_labels:
            mask = bucket == lbl
            fr = fwd_resid[mask].dropna()
            fs = fwd_spread[mask].dropna()
            n = len(fr)
            avg_fr = fr.mean() if n > 0 else float('nan')
            avg_fs = fs.mean() if n > 0 else float('nan')
            if lbl in neutral_buckets or n == 0:
                hit = float('nan')
            elif lbl in pos_buckets:
                hit = (fr < 0).mean() * 100
            else:
                hit = (fr > 0).mean() * 100
            rows.append({'Z-Bucket': lbl, 'Count': n,
                         'Avg Fwd Resid Chg': avg_fr,
                         'Avg Fwd Spread Chg': avg_fs,
                         'Hit Rate': hit})
        results[w] = pd.DataFrame(rows).set_index('Z-Bucket')
    return results

fwd_results = forward_return_analysis(
    decomp, df_spreads, FOCUS_COUNTRY, FWD_WINDOWS,
    ZSCORE_BINS, ZSCORE_LABELS, SD_WINDOW, LOOKBACK_START
)

# print tables
for w, tbl in fwd_results.items():
    print(f'\n=== Forward Return Analysis: {FOCUS_COUNTRY} | {TENOR} | {w}d Horizon ===\n')
    print(tbl.to_string(
        float_format=lambda x: f'{x:.1f}',
        formatters={
            'Count': lambda x: f'{int(x):d}',
            'Hit Rate': lambda x: f'{x:.0f}%' if not pd.isna(x) else 'n/a',
        }
    ))

# 2x2 bar chart grid
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()
bar_colors = {'>1.65': 'red', '1.25:1.65': 'salmon',
              '-1.65:-1.25': 'steelblue', '<-1.65': 'navy',
              '-1.25:0': 'lightgray', '0:1.25': 'lightgray'}

for ax, w in zip(axes, FWD_WINDOWS):
    tbl = fwd_results[w]
    vals = tbl['Avg Fwd Resid Chg']
    colors = [bar_colors.get(lbl, 'gray') for lbl in vals.index]
    bars = ax.bar(range(len(vals)), vals.values, color=colors)
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels(vals.index, rotation=20, fontsize=8)
    ax.axhline(0, color='black', linewidth=0.7)
    ax.set_title(f'{w}d Horizon')
    ax.set_ylabel('Avg Fwd Resid Chg (bps)')
    for bar, val in zip(bars, vals.values):
        if not pd.isna(val):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + (0.3 if val >= 0 else -0.5),
                    f'{val:.1f}', ha='center',
                    va='bottom' if val >= 0 else 'top', fontsize=8)

plt.suptitle(f'Forward Residual Change by Z-Score Bucket — {FOCUS_COUNTRY} ({TENOR})', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 18 — Conditional Signal (Momentum Filter)
# extreme buckets only: '<-1.65' and '>1.65'
resid_fc = decomp['residuals'][FOCUS_COUNTRY].copy()
if LOOKBACK_START is not None:
    resid_fc = resid_fc[resid_fc.index >= LOOKBACK_START]
actual_fc = df_spreads[FOCUS_COUNTRY].reindex(resid_fc.index)
roll_sd_fc = resid_fc.rolling(SD_WINDOW).std()
zscore_fc = resid_fc / roll_sd_fc

# PC1 20d rolling momentum aligned to same index
pc1_mom = roll_factor[20]['PC1'].reindex(resid_fc.index)

print(f'=== Conditional Signal: {FOCUS_COUNTRY} | Momentum Filter ===\n')
header = f'{"Sub-group":<30}' + ''.join([f'{w:>8}d Chg  HR%' for w in FWD_WINDOWS])
print(header)
print('-' * (30 + 20 * len(FWD_WINDOWS)))

for extreme_lbl, z_sign in [('>1.65', 1), ('<-1.65', -1)]:
    # mask for this extreme bucket
    if z_sign == 1:
        extreme_mask = zscore_fc > 1.65
    else:
        extreme_mask = zscore_fc < -1.65
    # high conviction: PC1 momentum has same sign as z (corrective = same sign as z for negative z,
    # or negative momentum for positive z — i.e. momentum is opposing the dislocation direction)
    # "same sign as z-score" means momentum is trending in the direction that CREATED the dislocation
    # "corrective" means momentum has OPPOSITE sign to dislocation, i.e. factor is reverting
    # Per spec: sub-group A = 20d PC1 roll sum has SAME sign as z-score
    if z_sign == 1:
        hi_conv_mask = extreme_mask & (pc1_mom > 0)
        lo_conv_mask = extreme_mask & (pc1_mom <= 0)
    else:
        hi_conv_mask = extreme_mask & (pc1_mom < 0)
        lo_conv_mask = extreme_mask & (pc1_mom >= 0)
    for sub_label, mask in [('High conviction (same-sign mom)', hi_conv_mask),
                             ('Low conviction (opp-sign mom)', lo_conv_mask)]:
        label_str = f'  [{extreme_lbl}] {sub_label}'
        row_str = f'{label_str:<30}'
        n_obs = mask.sum()
        row_str += f'  n={n_obs:<4}'
        for w in FWD_WINDOWS:
            fwd_resid = resid_fc.shift(-w) - resid_fc
            fr = fwd_resid[mask].dropna()
            if len(fr) == 0:
                row_str += f'{"n/a":>14}'
                continue
            avg_fr = fr.mean()
            if z_sign == 1:
                hit = (fr < 0).mean() * 100
            else:
                hit = (fr > 0).mean() * 100
            row_str += f'  {avg_fr:>6.1f}bps {hit:>4.0f}%'
        print(row_str)
    print()

In [ ]:
# Cell 19 — Signal Summary
resid_now = decomp['residuals'][FOCUS_COUNTRY].dropna()
roll_sd_now = resid_now.rolling(SD_WINDOW).std()
zscore_now = (resid_now / roll_sd_now).dropna()
current_z = zscore_now.iloc[-1]
current_resid = resid_now.iloc[-1]

# current bucket
current_bucket = pd.cut(
    pd.Series([current_z]), bins=ZSCORE_BINS, labels=ZSCORE_LABELS
).iloc[0]

# PC1 and PC2 current 20d momentum
pc1_mom_now = roll_factor[20]['PC1'].iloc[-1]
pc2_mom_now = roll_factor[20]['PC2'].iloc[-1] if 'PC2' in roll_factor[20].columns else float('nan')

# historical stats for current bucket from 20d horizon table (primary reference)
ref_horizon = 20
tbl_ref = fwd_results[ref_horizon]
neutral_buckets = {'-1.25:0', '0:1.25'}

if current_bucket in tbl_ref.index:
    avg_fr_ref = tbl_ref.loc[current_bucket, 'Avg Fwd Resid Chg']
    hit_ref = tbl_ref.loc[current_bucket, 'Hit Rate']
    count_ref = tbl_ref.loc[current_bucket, 'Count']
else:
    avg_fr_ref, hit_ref, count_ref = float('nan'), float('nan'), 0

# momentum conviction — only meaningful for extreme buckets
pos_buckets = {'>1.65', '1.25:1.65'}
neg_buckets = {'<-1.65', '-1.65:-1.25'}

def conviction_label(bucket, pc1_mom):
    if bucket in pos_buckets:
        return 'high conviction (corrective)' if pc1_mom < 0 else 'low conviction (reinforcing)'
    if bucket in neg_buckets:
        return 'high conviction (corrective)' if pc1_mom > 0 else 'low conviction (reinforcing)'
    return 'neutral zone'

conviction = conviction_label(str(current_bucket), pc1_mom_now)

# --- print summary ---
print(f'{FOCUS_COUNTRY} residual z-score is {current_z:+.2f} (bucket {current_bucket}).')
print(f'Current residual: {current_resid:.1f} bps.')

if str(current_bucket) not in neutral_buckets:
    if not pd.isna(avg_fr_ref):
        dir_word = 'compression' if avg_fr_ref < 0 else 'widening'
        hr_str = f'{hit_ref:.0f}% hit rate' if not pd.isna(hit_ref) else 'no hit rate (neutral)'
        print(
            f'Historically (n={int(count_ref)}), avg residual {dir_word} of '
            f'{avg_fr_ref:.1f} bps over {ref_horizon}d with {hr_str}.'
        )
    # all horizons snapshot
    print(f'\nForward residual change by horizon (avg bps | hit rate):')
    for w in FWD_WINDOWS:
        row = fwd_results[w].loc[current_bucket] if current_bucket in fwd_results[w].index else None
        if row is not None and not pd.isna(row['Avg Fwd Resid Chg']):
            hr_str = f'{row["Hit Rate"]:.0f}%' if not pd.isna(row['Hit Rate']) else 'n/a'
            print(f'  {w:>3}d:  {row["Avg Fwd Resid Chg"]:>6.1f} bps  |  {hr_str}')
else:
    print('Residual is in a neutral zone — no strong directional expectation.')

pc2_str = f'  PC2 20d momentum: {pc2_mom_now:.2f} bps.' if not pd.isna(pc2_mom_now) else ''
print(f'\nPC1 20d momentum: {pc1_mom_now:.2f} bps.{pc2_str}')
print(f'Momentum filter: {conviction}.')

In [ ]:
# Part 3A — Backtest parameters
ENTRY_THRESHOLD = 1.65
EXIT_ZSCORE = 0.5
FIXED_HOLD = [20, 40]
MAX_HOLD = 60
STOP_LOSS = 30
BACKTEST_COUNTRY = 'Peru'

In [ ]:
# Cell 20 — Backtest Engine
import pandas as pd
import numpy as np

def run_backtest(residuals, zscore, entry_thresh, exit_rule, exit_param, max_hold, stop_loss):
    resid = residuals.copy()
    zs = zscore.copy()
    # align on common valid index
    valid = resid.notna() & zs.notna()
    resid = resid[valid]
    zs = zs[valid]
    dates = resid.index
    trades = []
    in_trade = False
    entry_i = None
    entry_resid = None
    entry_z = None
    direction = None
    for i, dt in enumerate(dates):
        cur_r = resid.iloc[i]
        cur_z = zs.iloc[i]
        if in_trade:
            days_held = i - entry_i
            unreal_pnl = direction * (cur_r - entry_resid)
            exit_reason = None
            # priority 1: stop-loss
            if unreal_pnl < -stop_loss:
                exit_reason = 'stop_loss'
            # priority 2: max hold
            elif days_held >= max_hold:
                exit_reason = 'max_hold'
            # priority 3: rule-specific
            elif exit_rule == 'zero_cross':
                if direction * cur_r <= 0:
                    exit_reason = 'signal'
            elif exit_rule == 'partial':
                if abs(cur_z) < exit_param:
                    exit_reason = 'signal'
            elif exit_rule == 'fixed':
                if days_held >= exit_param:
                    exit_reason = 'signal'
            if exit_reason is not None:
                pnl = direction * (cur_r - entry_resid)
                trades.append({
                    'entry_date': dates[entry_i],
                    'exit_date': dt,
                    'direction': direction,
                    'entry_resid': entry_resid,
                    'exit_resid': cur_r,
                    'entry_z': entry_z,
                    'pnl_bps': pnl,
                    'hold_days': days_held,
                    'exit_reason': exit_reason,
                })
                in_trade = False
        if not in_trade:
            if cur_z > entry_thresh:
                in_trade = True
                entry_i = i
                entry_resid = cur_r
                entry_z = cur_z
                direction = -1
            elif cur_z < -entry_thresh:
                in_trade = True
                entry_i = i
                entry_resid = cur_r
                entry_z = cur_z
                direction = 1
    # close any open trade at end
    if in_trade:
        i = len(dates) - 1
        cur_r = resid.iloc[i]
        pnl = direction * (cur_r - entry_resid)
        trades.append({
            'entry_date': dates[entry_i],
            'exit_date': dates[i],
            'direction': direction,
            'entry_resid': entry_resid,
            'exit_resid': cur_r,
            'entry_z': entry_z,
            'pnl_bps': pnl,
            'hold_days': i - entry_i,
            'exit_reason': 'forced_close',
        })
    if not trades:
        cols = ['entry_date', 'exit_date', 'direction', 'entry_resid',
                'exit_resid', 'entry_z', 'pnl_bps', 'hold_days', 'exit_reason']
        return pd.DataFrame(columns=cols)
    return pd.DataFrame(trades)

In [ ]:
# Cell 21 — Run All Three Exit Rules
resid_bt = decomp['residuals'][BACKTEST_COUNTRY].dropna()
roll_sd_bt = resid_bt.rolling(SD_WINDOW).std()
zscore_bt = (resid_bt / roll_sd_bt).dropna()
resid_bt = resid_bt.reindex(zscore_bt.index)

bt_strategies = {
    'Zero Cross': run_backtest(
        resid_bt, zscore_bt, ENTRY_THRESHOLD, 'zero_cross', None, MAX_HOLD, STOP_LOSS),
    'Partial (z<0.5)': run_backtest(
        resid_bt, zscore_bt, ENTRY_THRESHOLD, 'partial', EXIT_ZSCORE, MAX_HOLD, STOP_LOSS),
    'Fixed 20d': run_backtest(
        resid_bt, zscore_bt, ENTRY_THRESHOLD, 'fixed', 20, MAX_HOLD, STOP_LOSS),
    'Fixed 40d': run_backtest(
        resid_bt, zscore_bt, ENTRY_THRESHOLD, 'fixed', 40, MAX_HOLD, STOP_LOSS),
}

for name, trades in bt_strategies.items():
    print(f'{name}: {len(trades)} trades')

In [ ]:
# Cell 22 — Backtest Performance Summary
def compute_metrics(trades):
    if len(trades) == 0:
        return {}
    pnl = trades['pnl_bps'].values
    wins = pnl[pnl > 0]
    losses = pnl[pnl <= 0]
    cum = np.cumsum(pnl)
    # max drawdown from cumulative P&L
    peak = np.maximum.accumulate(cum)
    dd = cum - peak
    max_dd = dd.min()
    # annualised trade count: use calendar days between first entry and last exit
    span_days = (trades['exit_date'].max() - trades['entry_date'].min()).days
    tpy = len(trades) / (span_days / 365.25) if span_days > 0 else float('nan')
    n = len(trades)
    reasons = trades['exit_reason'].value_counts(normalize=True) * 100
    return {
        'Total Trades': n,
        'Win Rate %': (pnl > 0).mean() * 100,
        'Avg P&L (bps)': pnl.mean(),
        'Avg Win (bps)': wins.mean() if len(wins) else float('nan'),
        'Avg Loss (bps)': losses.mean() if len(losses) else float('nan'),
        'Profit Factor': wins.sum() / abs(losses.sum()) if len(losses) else float('nan'),
        'Avg Hold (days)': trades['hold_days'].mean(),
        'Max Drawdown (bps)': max_dd,
        'Total P&L (bps)': cum[-1],
        'Trades/Year': tpy,
        'Exit: Signal %': reasons.get('signal', 0),
        'Exit: Stop %': reasons.get('stop_loss', 0),
        'Exit: MaxHold %': reasons.get('max_hold', 0),
    }

metrics_rows = {name: compute_metrics(t) for name, t in bt_strategies.items()}
metrics_df = pd.DataFrame(metrics_rows)

int_metrics = {'Total Trades'}
pct_metrics = {'Win Rate %', 'Exit: Signal %', 'Exit: Stop %', 'Exit: MaxHold %'}

print(f'=== Backtest Summary: {BACKTEST_COUNTRY} | {TENOR} ===\n')
for metric, row in metrics_df.iterrows():
    line = f'{metric:<22}'
    for val in row:
        if metric in int_metrics:
            line += f'  {int(val):>10}'
        elif metric in pct_metrics:
            line += f'  {val:>9.1f}%'
        else:
            line += f'  {val:>10.1f}'
    print(line)
print()
print(f'{"":22}' + ''.join(f'  {n:>10}' for n in metrics_df.columns))

In [ ]:
# Cell 23 — Cumulative P&L Chart
import matplotlib.pyplot as plt

strategy_colors = {
    'Zero Cross': 'steelblue',
    'Partial (z<0.5)': 'darkorange',
    'Fixed 20d': 'green',
    'Fixed 40d': 'purple',
}

fig, ax = plt.subplots(figsize=(14, 6))
for name, trades in bt_strategies.items():
    if len(trades) == 0:
        continue
    t = trades.sort_values('exit_date')
    dates_plot = [t['entry_date'].iloc[0]] + list(t['exit_date'])
    cum_pnl = [0.0] + list(np.cumsum(t['pnl_bps'].values))
    ax.step(dates_plot, cum_pnl, where='post',
            label=name, color=strategy_colors.get(name, 'black'), linewidth=1.5)

ax.axhline(0, color='black', linewidth=0.7, linestyle='--')
ax.set_ylabel('Cumulative P&L (bps)')
ax.set_title(
    f'Cumulative P&L — {BACKTEST_COUNTRY} {TENOR} Spread Dislocation '
    f'({LOOKBACK_START} onwards)'
)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 24 — Trade Detail Chart
# pick best strategy by total cumulative P&L
best_name = max(
    bt_strategies,
    key=lambda n: np.sum(bt_strategies[n]['pnl_bps'].values) if len(bt_strategies[n]) > 0 else -np.inf
)
best_trades = bt_strategies[best_name].sort_values('entry_date').reset_index(drop=True)

# build cumulative P&L step series at trade resolution
cum_vals = np.cumsum(best_trades['pnl_bps'].values)
cum_dates = list(best_trades['exit_date'])
cum_start = [best_trades['entry_date'].iloc[0]]
cum_pnl_step = [0.0] + list(cum_vals)
cum_dates_step = cum_start + cum_dates

# drawdown series on trade-level
peak_arr = np.maximum.accumulate(cum_vals)
dd_arr = cum_vals - peak_arr

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=False)

# top: residual time series with entries/exits and shaded holding periods
ax1.plot(resid_bt.index, resid_bt.values, color='black', linewidth=0.9, zorder=2)
ax1.axhline(0, color='gray', linewidth=0.5)

for _, tr in best_trades.iterrows():
    color = 'green' if tr['pnl_bps'] > 0 else 'red'
    ax1.axvspan(tr['entry_date'], tr['exit_date'], alpha=0.12, color=color, zorder=1)
    # entry marker
    marker = '^' if tr['direction'] == 1 else 'v'
    mcolor = 'green' if tr['direction'] == 1 else 'red'
    ax1.plot(tr['entry_date'], tr['entry_resid'], marker=marker, color=mcolor,
             markersize=8, zorder=3)
    # exit marker
    ax1.plot(tr['exit_date'], tr['exit_resid'], marker='x', color='black',
             markersize=7, markeredgewidth=1.5, zorder=3)

ax1.set_ylabel('Residual (bps)')
ax1.set_title(f'Trade Detail — {best_name} | {BACKTEST_COUNTRY} ({TENOR})')

# bottom: cumulative P&L with drawdown shading
ax2.step(cum_dates_step, cum_pnl_step, where='post', color='steelblue', linewidth=1.5)
ax2.axhline(0, color='black', linewidth=0.7, linestyle='--')

# shade drawdown periods
for i in range(len(dd_arr)):
    if dd_arr[i] < 0:
        x0 = best_trades['entry_date'].iloc[i]
        x1 = best_trades['exit_date'].iloc[i]
        y_top = peak_arr[i]
        y_bot = cum_vals[i]
        ax2.fill_between([x0, x1], [y_bot, y_bot], [y_top, y_top],
                         alpha=0.2, color='red', step=None)

ax2.set_ylabel('Cumulative P&L (bps)')
ax2.set_xlabel('Date')

plt.tight_layout()
plt.show()
print(f'Best strategy: {best_name}')

In [ ]:
# Cell 25 — Momentum-Conditioned Backtest
# re-use the best exit rule config
best_rule_map = {
    'Zero Cross': ('zero_cross', None),
    'Partial (z<0.5)': ('partial', EXIT_ZSCORE),
    'Fixed 20d': ('fixed', 20),
    'Fixed 40d': ('fixed', 40),
}
best_rule, best_param = best_rule_map[best_name]

# PC1 20d rolling momentum aligned to backtest index
pc1_mom_bt = roll_factor[20]['PC1'].reindex(zscore_bt.index)

# build momentum-filtered zscore and residual series
# Strategy A: only allow entry when PC1 momentum has corrective sign
# corrective for short (z > +1.65): PC1 mom > 0 (factor reverting upward → spread compressing)
# corrective for long  (z < -1.65): PC1 mom < 0 (factor reverting downward → spread widening)
# implement by masking the z-score to zero where momentum is not corrective
zscore_mom_confirmed = zscore_bt.copy()
# when z > thresh but pc1 mom is not positive: suppress entry by zeroing out the spike
suppress_short = (zscore_bt > ENTRY_THRESHOLD) & (pc1_mom_bt <= 0)
suppress_long = (zscore_bt < -ENTRY_THRESHOLD) & (pc1_mom_bt >= 0)
zscore_mom_confirmed[suppress_short | suppress_long] = 0.0

trades_mom_confirmed = run_backtest(
    resid_bt, zscore_mom_confirmed, ENTRY_THRESHOLD,
    best_rule, best_param, MAX_HOLD, STOP_LOSS
)
trades_baseline = bt_strategies[best_name].copy()

mom_strategies = {
    f'{best_name} (baseline)': trades_baseline,
    f'{best_name} + Mom Filter': trades_mom_confirmed,
}

print(f'=== Momentum Filter Impact: {BACKTEST_COUNTRY} | {TENOR} ===\n')
mom_metrics = {name: compute_metrics(t) for name, t in mom_strategies.items()}
mom_df = pd.DataFrame(mom_metrics)
int_metrics = {'Total Trades'}
pct_metrics = {'Win Rate %', 'Exit: Signal %', 'Exit: Stop %', 'Exit: MaxHold %'}
for metric, row in mom_df.iterrows():
    line = f'{metric:<22}'
    for val in row:
        if metric in int_metrics:
            line += f'  {int(val):>22}'
        elif metric in pct_metrics:
            line += f'  {val:>21.1f}%'
        else:
            line += f'  {val:>22.1f}'
    print(line)
print()
print(f'{"":22}' + ''.join(f'  {n:>22}' for n in mom_df.columns))

# overlay cumulative P&L
fig, ax = plt.subplots(figsize=(14, 5))
for (name, trades), color in zip(mom_strategies.items(), ['steelblue', 'darkorange']):
    if len(trades) == 0:
        continue
    t = trades.sort_values('exit_date')
    d = [t['entry_date'].iloc[0]] + list(t['exit_date'])
    c = [0.0] + list(np.cumsum(t['pnl_bps'].values))
    ax.step(d, c, where='post', label=name, color=color, linewidth=1.5)
ax.axhline(0, color='black', linewidth=0.7, linestyle='--')
ax.set_ylabel('Cumulative P&L (bps)')
ax.set_title(f'=== Momentum Filter Impact: {BACKTEST_COUNTRY} | {TENOR} ===')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 26 — Trade Statistics by Direction
longs = best_trades[best_trades['direction'] == 1]
shorts = best_trades[best_trades['direction'] == -1]

dir_strategies = {
    'Long (z<-1.65)': longs,
    'Short (z>+1.65)': shorts,
    'All': best_trades,
}

print(f'=== Performance by Direction: {BACKTEST_COUNTRY} | {TENOR} ===\n')
dir_metrics = {name: compute_metrics(t) for name, t in dir_strategies.items()}
dir_df = pd.DataFrame(dir_metrics)
for metric, row in dir_df.iterrows():
    line = f'{metric:<22}'
    for val in row:
        if metric in int_metrics:
            line += f'  {int(val):>18}'
        elif metric in pct_metrics:
            line += f'  {val:>17.1f}%'
        else:
            line += f'  {val:>18.1f}'
    print(line)
print()
print(f'{"":22}' + ''.join(f'  {n:>18}' for n in dir_df.columns))

In [ ]:
# Cell 27 — Rolling 1Y Performance
import matplotlib.pyplot as plt
import numpy as np

trades_sorted = best_trades.sort_values('exit_date').reset_index(drop=True)
n_trades = len(trades_sorted)

# for each trade, look back 252 calendar days from its exit_date
window_days = 252

roll_wr = []
roll_avg_pnl = []
roll_n = []
roll_dates = []

for i in range(n_trades):
    exit_dt = trades_sorted['exit_date'].iloc[i]
    cutoff = exit_dt - pd.Timedelta(days=window_days)
    mask = (trades_sorted['exit_date'] <= exit_dt) & (trades_sorted['exit_date'] > cutoff)
    sub = trades_sorted[mask]
    if len(sub) < 3:
        continue
    pnl = sub['pnl_bps'].values
    roll_wr.append((pnl > 0).mean() * 100)
    roll_avg_pnl.append(pnl.mean())
    roll_n.append(len(sub))
    roll_dates.append(exit_dt)

roll_dates = pd.to_datetime(roll_dates)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(roll_dates, roll_wr, color='steelblue', linewidth=1.3)
axes[0].axhline(50, color='gray', linewidth=0.7, linestyle='--')
axes[0].set_ylabel('Win Rate (%)')
axes[0].set_title(f'Rolling 1Y Performance — {BACKTEST_COUNTRY} ({TENOR}) | {best_name}')

axes[1].plot(roll_dates, roll_avg_pnl, color='darkorange', linewidth=1.3)
axes[1].axhline(0, color='gray', linewidth=0.7, linestyle='--')
axes[1].set_ylabel('Avg P&L / Trade (bps)')

axes[2].bar(roll_dates, roll_n, color='slategray', width=5)
axes[2].set_ylabel('# Trades (trailing 252d)')
axes[2].set_xlabel('Date')

plt.tight_layout()
plt.show()

In [ ]:
# Part 3B — Directional signal parameters
FAST_WINDOW = 10
SLOW_WINDOW = 40
SCORE_ZSCORE_WINDOW = 252
LEVEL_THRESH = 1.5
DIRECTION_COUNTRY = 'Peru'
DIR_FWD_WINDOWS = [5, 10, 20, 40]

In [ ]:
# Cell 28 — Factor Stance Indicator
import pandas as pd
import numpy as np

pc1_scores = decomp['scores_full']['PC1']
pc1_fr = factor_returns['PC1']

# component 1: fast momo
fast_sum = pc1_fr.rolling(FAST_WINDOW).sum()
fast_signal = fast_sum.apply(lambda x: 1 if x > 0 else -1)

# component 2: slow momo
slow_sum = pc1_fr.rolling(SLOW_WINDOW).sum()
slow_signal = slow_sum.apply(lambda x: 1 if x > 0 else -1)

# component 3: level mean-reversion
pc1_roll_mean = pc1_scores.rolling(SCORE_ZSCORE_WINDOW).mean()
pc1_roll_std = pc1_scores.rolling(SCORE_ZSCORE_WINDOW).std()
pc1_level_z = (pc1_scores - pc1_roll_mean) / pc1_roll_std
# sign convention: PC1 loadings are negative
# high PC1 score -> spreads compressed -> bearish (expect widening) -> -1
# low PC1 score  -> spreads wide        -> bullish (expect compression) -> +1
level_signal = pc1_level_z.apply(
    lambda x: -1 if x > LEVEL_THRESH else (1 if x < -LEVEL_THRESH else 0)
)

# composite: sum of three components, carry-forward on zero
raw_composite = fast_signal + slow_signal + level_signal
composite_raw = raw_composite.apply(lambda x: 1 if x > 0 else (-1 if x < 0 else 0))
stance = composite_raw.copy().astype(float)
# carry forward previous value when sum == 0
for i in range(1, len(stance)):
    if stance.iloc[i] == 0:
        stance.iloc[i] = stance.iloc[i - 1]
stance = stance.fillna(method='ffill').fillna(1)
stance = stance.astype(int)

# PC2 differentiation flag
if 'PC2' in decomp['scores_full'].columns:
    pc2_scores = decomp['scores_full']['PC2']
else:
    pc2_scores = decomp['scores_full'].iloc[:, 1] if decomp['scores_full'].shape[1] > 1 else pd.Series(0, index=pc1_scores.index)
pc2_roll_mean = pc2_scores.rolling(SCORE_ZSCORE_WINDOW).mean()
pc2_roll_std = pc2_scores.rolling(SCORE_ZSCORE_WINDOW).std()
pc2_level_z = (pc2_scores - pc2_roll_mean) / pc2_roll_std
pc2_diff_flag = pc2_level_z.abs() > LEVEL_THRESH

# current snapshot
cur_fast_val = fast_sum.iloc[-1]
cur_slow_val = slow_sum.iloc[-1]
cur_pc1_z = pc1_level_z.iloc[-1]
cur_stance = stance.iloc[-1]
cur_pc2_z = pc2_level_z.iloc[-1]
cur_diff = pc2_diff_flag.iloc[-1]

print('=== Factor Stance Snapshot ===')
print(f'Fast momo ({FAST_WINDOW}d sum):  {cur_fast_val:+.2f}  ->  signal {fast_signal.iloc[-1]:+d}')
print(f'Slow momo ({SLOW_WINDOW}d sum):  {cur_slow_val:+.2f}  ->  signal {slow_signal.iloc[-1]:+d}')
print(f'PC1 level z-score:   {cur_pc1_z:+.2f}  ->  signal {level_signal.iloc[-1]:+.0f}')
print(f'Composite stance:    {cur_stance:+d}  ({["Bearish/Short", "Neutral", "Bullish/Long"][cur_stance + 1]})')
print(f'PC2 level z-score:   {cur_pc2_z:+.2f}  ->  differentiation flag: {cur_diff}')
if cur_diff:
    print('  ** Differentiation regime: country selection likely matters more than direction **')

In [ ]:
# Cell 29 — Forward Spread Change by Stance (Signal Validation)
import matplotlib.pyplot as plt

spread_dir = df_spreads[DIRECTION_COUNTRY].copy()

# align all signals to common valid index
common_idx = spread_dir.dropna().index
fast_s = fast_signal.reindex(common_idx)
slow_s = slow_signal.reindex(common_idx)
comp_s = stance.reindex(common_idx)
spread_a = spread_dir.reindex(common_idx)

signal_variants = {
    'Fast Only': fast_s,
    'Slow Only': slow_s,
    'Composite': comp_s,
}

def stance_fwd_table(sig, spread, horizons):
    rows = []
    for st, label in [(1, 'Long (+1)'), (-1, 'Short (-1)')]:
        mask = sig == st
        row = {'Stance': label}
        for w in horizons:
            fwd_chg = spread.shift(-w) - spread
            subset = fwd_chg[mask].dropna()
            n = len(subset)
            avg = subset.mean() if n > 0 else float('nan')
            # long: good = compression = negative change; short: good = widening = positive change
            if n > 0:
                hit = (subset < 0).mean() * 100 if st == 1 else (subset > 0).mean() * 100
            else:
                hit = float('nan')
            row[f'{w}d Avg'] = avg
            row[f'{w}d Hit%'] = hit
            row[f'{w}d N'] = n
        rows.append(row)
    return pd.DataFrame(rows).set_index('Stance')

variant_tables = {}
for vname, vsig in signal_variants.items():
    tbl = stance_fwd_table(vsig, spread_a, DIR_FWD_WINDOWS)
    variant_tables[vname] = tbl
    print(f'=== Stance Signal Validation: {vname} | {DIRECTION_COUNTRY} | {TENOR} ===\n')
    header = f'{"":18}'
    for w in DIR_FWD_WINDOWS:
        header += f'  {w}d Avg  {w}d Hit%  {w}d N'
    print(header)
    for stance_lbl, row in tbl.iterrows():
        line = f'{stance_lbl:<18}'
        for w in DIR_FWD_WINDOWS:
            avg = row[f'{w}d Avg']
            hit = row[f'{w}d Hit%']
            n = int(row[f'{w}d N'])
            line += f'  {avg:>7.1f}  {hit:>7.1f}%  {n:>4d}'
        print(line)
    print()

# 2x2 bar chart: avg forward spread change per stance
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()
bar_width = 0.25
x = np.arange(len(signal_variants))
xtick_labels = list(signal_variants.keys())
stance_labels = ['Long (+1)', 'Short (-1)']
stance_colors = ['steelblue', 'tomato']

for ax, w in zip(axes, DIR_FWD_WINDOWS):
    for si, (st_lbl, color) in enumerate(zip(stance_labels, stance_colors)):
        vals = []
        for vname in signal_variants:
            row = variant_tables[vname].loc[st_lbl]
            vals.append(row[f'{w}d Avg'])
        offset = (si - 0.5) * bar_width
        bars = ax.bar(x + offset, vals, width=bar_width, label=st_lbl, color=color, alpha=0.8)
        for bar, val in zip(bars, vals):
            if not np.isnan(val):
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + (0.3 if val >= 0 else -0.5),
                        f'{val:.1f}', ha='center',
                        va='bottom' if val >= 0 else 'top', fontsize=7)
    ax.axhline(0, color='black', linewidth=0.7)
    ax.set_xticks(x)
    ax.set_xticklabels(xtick_labels, fontsize=9)
    ax.set_title(f'{w}d Horizon')
    ax.set_ylabel('Avg Fwd Spread Chg (bps)')
    if w == DIR_FWD_WINDOWS[0]:
        ax.legend(fontsize=9)

plt.suptitle(f'Forward Spread Change by Stance — {DIRECTION_COUNTRY} ({TENOR})', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 30 — Directional Backtest
import pandas as pd
import numpy as np

spread_dir_bt = df_spreads[DIRECTION_COUNTRY].copy()
daily_spread_chg = spread_dir_bt.diff()

# align signals to spread change index
all_idx = daily_spread_chg.dropna().index
fast_a = fast_signal.reindex(all_idx)
slow_a = slow_signal.reindex(all_idx)
comp_a = stance.reindex(all_idx)
dsc = daily_spread_chg.reindex(all_idx)

# daily P&L: -stance * spread_change (long local = benefit from compression)
pnl_fast = -fast_a * dsc
pnl_slow = -slow_a * dsc
pnl_comp = -comp_a * dsc

def dir_metrics(pnl_series, sig_series):
    pnl = pnl_series.dropna().values
    sig = sig_series.reindex(pnl_series.dropna().index).values
    cum = np.cumsum(pnl)
    # max drawdown
    peak = np.maximum.accumulate(cum)
    dd = cum - peak
    max_dd = dd.min()
    # sharpe
    mean_d = pnl.mean()
    std_d = pnl.std()
    sharpe = (mean_d / std_d * np.sqrt(252)) if std_d > 0 else float('nan')
    # years
    idx = pnl_series.dropna().index
    years = (idx[-1] - idx[0]).days / 365.25
    ann_pnl = cum[-1] / years if years > 0 else float('nan')
    # stance breakdown
    pct_long = (sig == 1).mean() * 100
    pct_short = (sig == -1).mean() * 100
    pct_flat = (sig == 0).mean() * 100
    avg_long = pnl[sig == 1].mean() if (sig == 1).any() else float('nan')
    avg_short = pnl[sig == -1].mean() if (sig == -1).any() else float('nan')
    # stance changes: count consecutive sign flips
    n_changes = int((np.diff(sig) != 0).sum())
    # avg holding period: total days / number of changes
    avg_hold = len(sig) / (n_changes + 1) if n_changes >= 0 else float('nan')
    return {
        'Total P&L (bps)': cum[-1],
        'Ann P&L (bps/yr)': ann_pnl,
        'Ann Sharpe': sharpe,
        'Max Drawdown (bps)': max_dd,
        '% Long days': pct_long,
        '% Short days': pct_short,
        '% Flat days': pct_flat,
        'Avg Daily PnL Long': avg_long,
        'Avg Daily PnL Short': avg_short,
        'Stance Changes': n_changes,
        'Avg Hold (days)': avg_hold,
    }

dir_bt = {
    'Fast Only': dir_metrics(pnl_fast, fast_a),
    'Slow Only': dir_metrics(pnl_slow, slow_a),
    'Composite': dir_metrics(pnl_comp, comp_a),
}
dir_df = pd.DataFrame(dir_bt)

int_m = {'Stance Changes'}
pct_m = {'% Long days', '% Short days', '% Flat days'}

print(f'=== Directional Backtest: {DIRECTION_COUNTRY} | {TENOR} ===\n')
print(f'{"Metric":<24}' + ''.join(f'  {n:>14}' for n in dir_df.columns))
print('-' * (24 + 18 * len(dir_df.columns)))
for metric, row in dir_df.iterrows():
    line = f'{metric:<24}'
    for val in row:
        if metric in int_m:
            line += f'  {int(val):>14}'
        elif metric in pct_m:
            line += f'  {val:>13.1f}%'
        else:
            line += f'  {val:>14.1f}'
    print(line)

# store daily pnl series for downstream use
dir_daily_pnl = {'Fast Only': pnl_fast, 'Slow Only': pnl_slow, 'Composite': pnl_comp}

In [ ]:
# Cell 31 — Directional Signal Charts
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(3, 1, figsize=(14, 13), sharex=True)

# --- top: PC1 score level with level zones ---
ax = axes[0]
ax.plot(pc1_scores.index, pc1_scores.values, color='black', linewidth=1.0, zorder=3)
ax.plot(pc1_roll_mean.index, pc1_roll_mean.values, color='steelblue',
        linewidth=1.3, linestyle='--', label='252d rolling mean', zorder=3)
# shade bullish zone (z < -LEVEL_THRESH): low PC1 score = spreads wide
idx_dates = pc1_level_z.dropna().index
for i in range(1, len(idx_dates)):
    z_prev = pc1_level_z.loc[idx_dates[i - 1]]
    z_curr = pc1_level_z.loc[idx_dates[i]]
    if z_curr < -LEVEL_THRESH:
        ax.axvspan(idx_dates[i - 1], idx_dates[i], alpha=0.15, color='green', zorder=1)
    elif z_curr > LEVEL_THRESH:
        ax.axvspan(idx_dates[i - 1], idx_dates[i], alpha=0.15, color='red', zorder=1)
ax.set_ylabel('PC1 Score')
ax.set_title(f'Directional Signal Dashboard — {DIRECTION_COUNTRY} ({TENOR})')
bull_patch = mpatches.Patch(color='green', alpha=0.3, label=f'Bullish zone (z<-{LEVEL_THRESH})')
bear_patch = mpatches.Patch(color='red', alpha=0.3, label=f'Bearish zone (z>+{LEVEL_THRESH})')
ax.legend(handles=[bull_patch, bear_patch,
          plt.Line2D([], [], color='steelblue', linestyle='--', label='252d mean')],
          fontsize=8, loc='upper right')

# --- middle: fast and slow momo with agreement shading ---
ax = axes[1]
fast_plot = fast_sum.reindex(idx_dates)
slow_plot = slow_sum.reindex(idx_dates)
ax.plot(fast_plot.index, fast_plot.values, color='steelblue',
        linewidth=0.9, label=f'Fast {FAST_WINDOW}d')
ax.plot(slow_plot.index, slow_plot.values, color='black',
        linewidth=1.6, label=f'Slow {SLOW_WINDOW}d')
ax.axhline(0, color='gray', linewidth=0.7)
# shade when both positive (green) or both negative (red)
for i in range(1, len(idx_dates)):
    f_val = fast_plot.loc[idx_dates[i]]
    s_val = slow_plot.loc[idx_dates[i]]
    if f_val > 0 and s_val > 0:
        ax.axvspan(idx_dates[i - 1], idx_dates[i], alpha=0.12, color='green', zorder=1)
    elif f_val < 0 and s_val < 0:
        ax.axvspan(idx_dates[i - 1], idx_dates[i], alpha=0.12, color='red', zorder=1)
ax.set_ylabel('Rolling Sum of PC1 Returns')
ax.legend(fontsize=9)

# --- bottom: cumulative P&L for all three ---
ax = axes[2]
line_styles = {'Fast Only': 'dashed', 'Slow Only': 'dotted', 'Composite': 'solid'}
line_colors = {'Fast Only': 'steelblue', 'Slow Only': 'darkorange', 'Composite': 'black'}
for name, pnl_s in dir_daily_pnl.items():
    pnl_v = pnl_s.dropna()
    cum = np.cumsum(pnl_v.values)
    ax.plot(pnl_v.index, cum,
            color=line_colors[name], linestyle=line_styles[name],
            linewidth=1.5, label=name)
ax.axhline(0, color='gray', linewidth=0.7, linestyle='--')
ax.set_ylabel('Cumulative P&L (bps)')
ax.set_xlabel('Date')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 32 — Directional Performance by Regime
import pandas as pd
import numpy as np

pnl_comp_s = dir_daily_pnl['Composite'].dropna()
comp_sig_a = comp_a.reindex(pnl_comp_s.index)

# PC2 differentiation flag aligned
pc2_flag_a = pc2_diff_flag.reindex(pnl_comp_s.index).fillna(False)

# spread vol proxy: rolling 60d std of spread daily changes
spread_chg_a = daily_spread_chg.reindex(pnl_comp_s.index)
roll_vol_60 = spread_chg_a.rolling(60).std()
vol_median = roll_vol_60.dropna().median()
high_vol = roll_vol_60 >= vol_median
low_vol = roll_vol_60 < vol_median

def regime_stats(pnl, mask, label):
    sub = pnl[mask].dropna()
    if len(sub) == 0:
        return {'Regime': label, 'Avg Daily PnL': float('nan'),
                'Sharpe': float('nan'), '% of Days': 0.0, 'Cumul PnL Contrib': float('nan')}
    pct_days = mask.reindex(pnl.index).fillna(False).mean() * 100
    mean_d = sub.mean()
    std_d = sub.std()
    sharpe = (mean_d / std_d * np.sqrt(252)) if std_d > 0 else float('nan')
    return {
        'Regime': label,
        'Avg Daily PnL': mean_d,
        'Sharpe': sharpe,
        '% of Days': pct_days,
        'Cumul PnL Contrib': sub.sum(),
    }

normal_mask = ~pc2_flag_a
diff_mask = pc2_flag_a.astype(bool)
# vol masks need same index alignment
hv_mask = high_vol.reindex(pnl_comp_s.index).fillna(False).astype(bool)
lv_mask = low_vol.reindex(pnl_comp_s.index).fillna(False).astype(bool)

regime_rows = [
    regime_stats(pnl_comp_s, normal_mask, 'Normal (PC2 low)'),
    regime_stats(pnl_comp_s, diff_mask, 'Differentiation (PC2 high)'),
    regime_stats(pnl_comp_s, hv_mask, 'High Spread Vol'),
    regime_stats(pnl_comp_s, lv_mask, 'Low Spread Vol'),
]
regime_df = pd.DataFrame(regime_rows).set_index('Regime')

print(f'=== Directional Performance by Regime: {DIRECTION_COUNTRY} | {TENOR} ===\n')
print(regime_df.to_string(float_format=lambda x: f'{x:.2f}',
                           formatters={'% of Days': lambda x: f'{x:.1f}%'}))

In [ ]:
# Cell 33 — Combined Strategy Summary
import pandas as pd
import numpy as np

# --- RV signal readings (from Part 2) ---
resid_33 = decomp['residuals'][FOCUS_COUNTRY].dropna()
roll_sd_33 = resid_33.rolling(SD_WINDOW).std()
zscore_33 = (resid_33 / roll_sd_33).dropna()
current_z_33 = zscore_33.iloc[-1]
current_bucket_33 = pd.cut(
    pd.Series([current_z_33]), bins=ZSCORE_BINS, labels=ZSCORE_LABELS
).iloc[0]
neutral_buckets = {'-1.25:0', '0:1.25'}
pos_buckets_rv = {'>1.65', '1.25:1.65'}
neg_buckets_rv = {'<-1.65', '-1.65:-1.25'}

# historical forward stats for current bucket at 20d horizon
ref_h = 20
if current_bucket_33 in fwd_results[ref_h].index:
    rv_avg = fwd_results[ref_h].loc[current_bucket_33, 'Avg Fwd Resid Chg']
    rv_hit = fwd_results[ref_h].loc[current_bucket_33, 'Hit Rate']
else:
    rv_avg, rv_hit = float('nan'), float('nan')

# momentum conviction for RV
pc1_mom_33 = roll_factor[20]['PC1'].iloc[-1]
if str(current_bucket_33) in pos_buckets_rv:
    rv_conviction = 'high conviction' if pc1_mom_33 < 0 else 'low conviction'
elif str(current_bucket_33) in neg_buckets_rv:
    rv_conviction = 'high conviction' if pc1_mom_33 > 0 else 'low conviction'
else:
    rv_conviction = 'neutral'

# --- Directional signal readings (from Cell 28) ---
dir_fast_val = fast_sum.iloc[-1]
dir_slow_val = slow_sum.iloc[-1]
dir_fast_sig = int(fast_signal.iloc[-1])
dir_slow_sig = int(slow_signal.iloc[-1])
dir_pc1_z = float(pc1_level_z.iloc[-1])
dir_stance = int(stance.iloc[-1])
dir_diff = bool(pc2_diff_flag.iloc[-1])

stance_word = 'Bullish' if dir_stance == 1 else 'Bearish'
dir_fast_word = 'bullish (compressing)' if dir_fast_sig == 1 else 'bearish (widening)'
dir_slow_word = 'bullish (compressing)' if dir_slow_sig == 1 else 'bearish (widening)'

# --- RV narrative ---
rv_is_extreme = str(current_bucket_33) not in neutral_buckets
rv_cheap = str(current_bucket_33) in neg_buckets_rv   # wide residual -> cheap
rv_rich = str(current_bucket_33) in pos_buckets_rv    # tight residual -> rich

# --- Combined narrative ---
print('=' * 70)
print(f'COMBINED SIGNAL SUMMARY — {FOCUS_COUNTRY} | {TENOR}')
print('=' * 70)

print('\n--- RV Signal ---')
print(f'  Residual z-score: {current_z_33:+.2f}  (bucket: {current_bucket_33})')
if rv_is_extreme:
    dir_word = 'cheap vs region (wide residual)' if rv_cheap else 'rich vs region (tight residual)'
    print(f'  {FOCUS_COUNTRY} is {dir_word}.')
    if not np.isnan(rv_avg):
        compression = 'compression' if rv_avg < 0 else 'widening'
        hr_str = f'{rv_hit:.0f}% hit rate' if not np.isnan(rv_hit) else 'n/a'
        print(f'  Historical avg residual {compression}: {rv_avg:.1f} bps over {ref_h}d ({hr_str}).')
    print(f'  PC1 20d momentum: {pc1_mom_33:+.2f} bps  ->  {rv_conviction}.')
else:
    print(f'  Residual is in neutral zone. No strong RV dislocation signal.')

print('\n--- Directional Signal ---')
print(f'  Fast momo ({FAST_WINDOW}d): {dir_fast_val:+.2f}  ->  {dir_fast_word}')
print(f'  Slow momo ({SLOW_WINDOW}d): {dir_slow_val:+.2f}  ->  {dir_slow_word}')
print(f'  PC1 level z-score: {dir_pc1_z:+.2f}')
print(f'  Composite stance: {dir_stance:+d}  ({stance_word})')
if dir_diff:
    print(f'  ** PC2 differentiation flag is active — country-selection regime **')

print('\n--- Combined Interpretation ---')

if rv_is_extreme and rv_cheap and dir_stance == 1:
    print(
        f'  HIGH CONVICTION LONG {FOCUS_COUNTRY.upper()} LOCAL DURATION.\n'
        f'  Both idiosyncratic cheapness and regional momentum support spread\n'
        f'  compression. RV z-score {current_z_33:+.2f} ({rv_conviction}) aligns with\n'
        f'  {stance_word.lower()} directional stance. Favour overweight.'
    )
elif rv_is_extreme and rv_rich and dir_stance == -1:
    print(
        f'  HIGH CONVICTION SHORT {FOCUS_COUNTRY.upper()} LOCAL DURATION.\n'
        f'  Both idiosyncratic richness and bearish regional momentum point to\n'
        f'  spread widening. RV z-score {current_z_33:+.2f} ({rv_conviction}) aligns with\n'
        f'  {stance_word.lower()} directional stance. Favour underweight.'
    )
elif rv_is_extreme and rv_cheap and dir_stance == -1:
    print(
        f'  CONFLICTING SIGNALS — {FOCUS_COUNTRY} CHEAP BUT REGIONAL TREND IS BEARISH.\n'
        f'  RV signal suggests {FOCUS_COUNTRY} is undervalued vs region (z={current_z_33:+.2f},\n'
        f'  {rv_conviction}), but regional spreads are still widening (directional stance bearish).\n'
        f'  Consider waiting for directional signal to turn before entering,\n'
        f'  or size position smaller to reflect the conflicting regime.'
    )
elif rv_is_extreme and rv_rich and dir_stance == 1:
    print(
        f'  CONFLICTING SIGNALS — {FOCUS_COUNTRY} RICH BUT REGIONAL TREND IS BULLISH.\n'
        f'  RV signal suggests {FOCUS_COUNTRY} is overvalued vs region (z={current_z_33:+.2f},\n'
        f'  {rv_conviction}), but regional spreads are compressing (directional stance bullish).\n'
        f'  The regional tailwind may continue to support tighter spreads despite richness.\n'
        f'  Avoid adding on the long; consider trimming if entering a differentiation regime.'
    )
elif not rv_is_extreme and dir_stance == 1:
    print(
        f'  NO RV DISLOCATION, BUT DIRECTIONAL SIGNAL IS BULLISH.\n'
        f'  {FOCUS_COUNTRY} residual is in neutral zone (z={current_z_33:+.2f}). No country-\n'
        f'  specific mispricing. However, regional directional stance is {stance_word.lower()}\n'
        f'  (fast {dir_fast_val:+.1f}, slow {dir_slow_val:+.1f}).\n'
        f'  Consider a generic LatAm local duration long without a {FOCUS_COUNTRY}-specific tilt.'
    )
elif not rv_is_extreme and dir_stance == -1:
    print(
        f'  NO RV DISLOCATION AND DIRECTIONAL SIGNAL IS BEARISH.\n'
        f'  {FOCUS_COUNTRY} residual is neutral (z={current_z_33:+.2f}) and regional momentum\n'
        f'  is {stance_word.lower()} (fast {dir_fast_val:+.1f}, slow {dir_slow_val:+.1f}).\n'
        f'  No attractive entry for local long. Hold underweight or flat LatAm duration.'
    )

if dir_diff:
    print(
        f'\n  NOTE: PC2 differentiation flag active (z={float(pc2_level_z.iloc[-1]):+.2f}).\n'
        f'  Cross-country divergence elevated — directional model less reliable.\n'
        f'  Prioritise RV country-selection over directional beta trades.'
    )
print()